# Phase 4 — Model Training

**Course**: CS5143 Natural Language Processing — Spring 2026, FAST-NUCES  
**Student**: Muhammad Azhar (24K-7606)

This notebook fine-tunes `xlm-roberta-base` on the joint EN+ES clinical NER dataset,
monitors per-epoch F1 on the English dev set, and saves the best checkpoint.

**Prerequisites**: Complete Phase 3 — `data/processed/joint/` must exist
(run `notebooks/02_preprocessing.ipynb` first).

**GPU strongly recommended.** Training 5 epochs on CPU may take several hours.
Use Google Colab (T4 GPU) or Kaggle (P100) if no local GPU is available.

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_from_disk
from transformers import DataCollatorForTokenClassification, EarlyStoppingCallback, Trainer, TrainingArguments

from dataset import id2label
from model import load_model, compute_metrics, compute_class_weights, WeightedLossTrainer
from utils import tokenizer

OUTPUT_DIR   = REPO_ROOT / 'outputs' / 'checkpoints'
RESULTS_DIR  = REPO_ROOT / 'outputs' / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING  : No GPU detected — training will be slow')

## 2. Load Dataset and Model

We load the joint EN+ES dataset saved in Phase 3, and initialise `xlm-roberta-base` with
a 7-class token classification head (`O`, `B/I-DIS`, `B/I-SYM`, `B/I-PRO`).

In [2]:
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed' / 'joint'
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f'Processed dataset not found at {PROCESSED_DIR}.\n'
        'Run notebooks/02_preprocessing.ipynb first.'
    )

joint_ds = load_from_disk(str(PROCESSED_DIR))
print('Dataset loaded:')
print(joint_ds)
print(f'\nTrain size : {len(joint_ds["train"])} sentences')
print(f'Dev size   : {len(joint_ds["validation"])} sentences')

model = load_model('xlm-roberta-base')
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel      : xlm-roberta-base')
print(f'Parameters : {total_params:,} total  |  {trainable:,} trainable')

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Dataset loaded:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20163
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2315
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2250
    })
})

Train size : 20163 sentences
Dev size   : 2315 sentences


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model      : xlm-roberta-base
Parameters : 277,458,439 total  |  277,458,439 trainable


## 3. Training Configuration

Key hyperparameters and their rationale (tuned for **Google Colab T4 16 GB**):

- **batch=32, accum=1 → effective batch=32**: T4 has 16 GB VRAM; XLM-R base at fp16 uses ~6 GB total
- **`lr_scheduler_type='cosine'`**: smooth annealing to near-zero for stable late-epoch convergence
- **`fp16=True`**: T4 Tensor Cores give ~2× throughput at half the VRAM footprint

**Three techniques stacked on top of the base Trainer:**

| Technique | Setting | Why |
|-----------|---------|-----|
| Class weights (alpha) | `max_ratio=2.0` | ~93% of tokens are `O`; 2× penalty for missing entities corrects imbalance without collapsing precision |
| Focal loss (gamma) | `focal_gamma=2.0` | Downweights easy examples (confident `O` predictions) and focuses gradient on hard clinical terms |
| LLRD | `llrd_decay=0.9` | Preserves pretrained multilingual representations in lower layers while letting upper layers adapt; classifier=2e-5, layer 0≈0.5e-5 |
| Early stopping | `patience=2` | Stops when dev F1 stalls; prevents overfitting past the optimal checkpoint |

In [ ]:
USE_FP16   = torch.cuda.is_available()
# T4 16 GB: batch=32 fp16 uses ~6 GB total — no accumulation needed.
BATCH_SIZE = 16
GRAD_ACCUM = 2
NUM_EPOCHS = 8   # increased from 5: more epochs needed with corrected class weights


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    # Cosine decay smoothly anneals LR to near-zero — better late-epoch
    # convergence than linear decay when training for 8 epochs.
    lr_scheduler_type='cosine',
    optim='adamw_torch',
    gradient_checkpointing=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='overall_f1',
    greater_is_better=True,
    fp16=USE_FP16,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    logging_steps=50,
    logging_dir=str(REPO_ROOT / 'outputs' / 'logs'),
    report_to='tensorboard',
    seed=42,
)

steps_per_epoch = len(joint_ds['train']) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Effective batch : {BATCH_SIZE * GRAD_ACCUM}')
print(f'Steps per epoch : ~{steps_per_epoch}')
print(f'Total steps     : ~{steps_per_epoch * NUM_EPOCHS}')
print(f'Mixed precision : {USE_FP16}')
print(f'LR scheduler    : cosine')

## 4. Train and Save Best Model

Training runs for up to 5 epochs. The HuggingFace `Trainer` evaluates on the English dev set
after every epoch and keeps the checkpoint with the highest `overall_f1`.

In [ ]:
# Launch TensorBoard to monitor training in real time.
# Works in Google Colab and local Jupyter — run this BEFORE trainer.train().
%load_ext tensorboard
%tensorboard --logdir {str(REPO_ROOT / 'outputs' / 'logs')}

In [ ]:
# ── Technique 1: Class weights (max_ratio=2×) ─────────────────────────────
# Corrects O-class dominance (~93% of tokens) without over-penalising FN vs FP.
MAX_WEIGHT_RATIO = 2.0
class_weights = compute_class_weights(joint_ds['train'], max_ratio=MAX_WEIGHT_RATIO)
print(f'Class weights (capped at {MAX_WEIGHT_RATIO}× min):')
for i, w in enumerate(class_weights.tolist()):
    print(f'  {id2label[i]:<8} {w:.3f}')

# ── Technique 2: Focal loss (γ=2) ─────────────────────────────────────────
# FL = -(1-p_t)^γ · log(p_t). Downweights easy tokens (confident O predictions)
# and focuses gradient signal on hard, ambiguous clinical terms.
# Combined with class weights (alpha): corrects imbalance AND focuses on hard cases.
FOCAL_GAMMA = 0.0

# ── Technique 3: LLRD (decay=0.9) ─────────────────────────────────────────
# Exponentially smaller LR for lower encoder layers to preserve pretrained
# multilingual representations while letting top layers adapt to clinical NER.
# classifier: base_lr=2e-5 | layer 11: 1.8e-5 | layer 0: 0.5e-5 | embeddings: 0.5e-5
LLRD_DECAY = 0.9

trainer = WeightedLossTrainer(
    class_weights=class_weights,
    focal_gamma=FOCAL_GAMMA,
    llrd_decay=LLRD_DECAY,
    model=model,
    args=training_args,
    train_dataset=joint_ds['train'],
    eval_dataset=joint_ds['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('\nStarting training ...')
trainer.train()

best_model_path = OUTPUT_DIR / 'best_model'
trainer.save_model(str(best_model_path))
print(f'\nBest model saved to {best_model_path}')

## 5. Training Curves

Plot training loss and dev F1 per epoch to diagnose underfitting/overfitting and confirm
that F1 is improving. A healthy run shows decreasing loss and increasing dev F1.

In [ ]:
log_history = pd.DataFrame(trainer.state.log_history)

train_log = log_history.dropna(subset=['loss']).copy()
eval_log  = log_history.dropna(subset=['eval_overall_f1']).copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(train_log['step'], train_log['loss'], color='#4C72B0', linewidth=1.5)
ax1.set(xlabel='Step', ylabel='Loss', title='Training Loss')
ax1.spines[['top', 'right']].set_visible(False)

ax2.plot(eval_log['epoch'], eval_log['eval_overall_f1'],
         marker='o', color='#55A868', linewidth=2, markersize=7)
ax2.axhline(0.75, color='red', linestyle='--', linewidth=1, label='Target F1 = 0.75')
ax2.set(xlabel='Epoch', ylabel='F1 (entity-level)', title='Dev F1 per Epoch')
ax2.legend()
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
curve_path = RESULTS_DIR / 'training_curves.png'
plt.savefig(str(curve_path), dpi=150, bbox_inches='tight')
print(f'Saved: {curve_path}')
plt.show()

best_f1 = eval_log['eval_overall_f1'].max()
best_ep = eval_log.loc[eval_log['eval_overall_f1'].idxmax(), 'epoch']
print(f'\nBest dev F1 : {best_f1:.4f}  (epoch {best_ep:.0f})')

## 6. Quick Dev Evaluation

Run seqeval on the dev set to get a per-entity breakdown before the full test evaluation
in Phase 5.

In [ ]:
from seqeval.metrics import classification_report

preds_out   = trainer.predict(joint_ds['validation'])
logits, labels = preds_out.predictions, preds_out.label_ids
predictions = np.argmax(logits, axis=2)

true_labels = [[id2label[l] for l in seq if l != -100] for seq in labels]
true_preds  = [[id2label[p] for (p, l) in zip(ps, ls) if l != -100]
               for ps, ls in zip(predictions, labels)]

print('=== Dev set results (EN) ===')
print(classification_report(true_labels, true_preds))

## Summary

**What was done in this phase:**
- Loaded `xlm-roberta-base` with a 7-class token classification head
- Fine-tuned on joint EN+ES clinical NER data for 5 epochs
- Monitored entity-level F1 on English dev set after each epoch
- Saved the best checkpoint to `outputs/checkpoints/best_model/`

**Next step**: Phase 5 — Evaluation & Error Analysis (`notebooks/04_evaluation.ipynb`)